In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
text1 = 'sentence-transformers embedding model'
text2 = 'sentence-transformers'

In [ ]:
embedding1 = model.encode(text1) #stored in vectordb
embedding2 = model.encode(text2)
#print(len(embedding))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
text1 = 'I love Artificial Intelligence'
text2 = 'I like Machine Learning'
emb1 = model.encode(text1)
emb2 = model.encode(text2)
print(cosine_similarity([emb1],[emb2]))

In [ ]:
pip install faiss-cpu

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning is a type of machine learning based on neural networks.",
    "Python is widely used for data science and machine learning.",
    "Hadoop is used for distributed storage and big data processing.",
    "Spark is faster than Hadoop for in-memory data processing."
]

In [ ]:
embedding=embedding_model.encode(documents)
embedding

In [ ]:
dim=embedding.shape[1]
dim

In [ ]:
index = faiss.IndexFlatL2(dim)  #L2-similarity search,fix the dimension
index.add(np.array(embedding))  #all the vector embeddings stored in index like object
print(index)

In [ ]:
generator = pipeline('text-generation',model='gpt2') #pipeline ->it is used for user query, text-generation->transformer model in that we used gpt2 model

In [ ]:
# ##step:5 RAG Function
# def rag_query(query,top_k=2):
#   #convert query to embedding
#   query_embedding=embedding_model.encode([query])
#   #search similar docs
#   distances,indices = index.search(np.array(query_embedding),top_k)
#   print(distances,indices)
#   print(indices[0])
#   #Retrieve relevant  documnets
#   retrieved_docs = [documents[i] for i in indices[0]]
#   #create agumented prompt
#   context = " ".join(retrieved_docs)
#   prompt = f"Answer the question based on the context.\n\n\nContext:{context}.\n\n\nQuery:{query}.\n\n\nAnswer:"
#   #Generator answer
#   result = generator(prompt,max_length=150,num_return_sequences=1)
#   return result[0]['generated_text']


# response=rag_query('what is deep learning?')
# print(response)

In [ ]:
# ----------------------------
# STEP 5: RAG Function
# ----------------------------
def rag_query(query, top_k=2):

    # Convert query to embedding
    query_embedding = embedding_model.encode([query])

    # Search similar docs
    distances, indices = index.search(np.array(query_embedding), top_k)
    print(distances,indices)
    print(indices[0])
    # Retrieve relevant documents
    retrieved_docs = [documents[i] for i in indices[0]]

    # # Create augmented prompt
    context = " ".join(retrieved_docs)
    prompt = f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: {query}\nAnswer:"

    # # Generate answer
    result = generator(prompt, max_length=150, num_return_sequences=1)

    return result[0]["generated_text"]


# ----------------------------
# STEP 6: Test Query
# ----------------------------
response = rag_query("What is deep learning?")
print(response)


#POS tagging in rag

In [ ]:
# ----------------------------
# Imports
# ----------------------------
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

# ----------------------------
# NLTK Downloads
# ----------------------------
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

# ----------------------------
# Lemmatizer
# ----------------------------
lemmatizer = WordNetLemmatizer()

# ----------------------------
# POS Mapping Function
# ----------------------------
def get_pos(tag):
    if tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# ----------------------------
# Query Preprocessing Function
# ----------------------------
def preprocess_query(query):
    tokens = word_tokenize(query)
    tagged = pos_tag(tokens)

    lemmatized = [
        lemmatizer.lemmatize(word, get_pos(pos))
        for word, pos in tagged
    ]

    return " ".join(lemmatized)

# ----------------------------
# STEP 5: RAG Function
# ----------------------------
def rag_query(query, top_k=2):

    # Convert query to embedding
    query_embedding = embedding_model.encode([query])

    # Search similar docs
    distances, indices = index.search(np.array(query_embedding), top_k)
    print(distances, indices)
    print(indices[0])

    # Retrieve relevant documents
    retrieved_docs = [documents[i] for i in indices[0]]

    # Create augmented prompt
    context = " ".join(retrieved_docs)
    prompt = f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: {query}\nAnswer:"

    # Generate answer
    result = generator(prompt, max_length=150, num_return_sequences=1)

    return result[0]["generated_text"]

# ----------------------------
# STEP 6: Test Query
# ----------------------------
user_query = "What is deep learning?"
processed_query = preprocess_query(user_query)

response = rag_query(processed_query)
print(response)